# 09 · ISR training set: an open-data cookbook

[ProSR](https://arxiv.org/abs/2609.02377) (Kim & Kim, KAIST-VICLab) curated
about **502 Umbra SLC** acquisitions into roughly **132k patches at 0.25 m**
as an ISR super-resolution benchmark
([KAIST-VICLab/ProSR](https://github.com/KAIST-VICLab/ProSR)). This notebook
is the **umbra-py** path to assemble a *similar* open-data corpus — filtered
search → size-check → georeferenced chips — without crawling S3 by hand and
without reimplementing ProSR or any diffusion / SR model.

Prefer `CatalogIndex.from_release()` for multi-filter searches (near-instant).
If the weekly snapshot is missing, run `umbra index fetch` once; the cells
below fall back to a tiny `UmbraCatalog` crawl only when needed.

The self-checking runnable path chips a **GEC** (streamed over HTTP, no
multi-GB download). SICD size-checks stay HEAD-only so CI / smoke runs stay
fast.

It needs the `load` extra (rasterio + numpy) for chipping:

```bash
pip install "umbra-py[load]"
```

> *Contains Umbra open data, licensed under CC BY 4.0.*

## 1 · Search VV SICD with polarization + incidence filters

ISR-style corpora usually pin polarization and look angle so patches are
comparable. `CatalogIndex.search` takes the same kwargs as
`UmbraCatalog.search` — here VV, 10–50° incidence, `product_types=["SICD"]`,
a small `limit`, and `max_per_task=1` for site diversity.

In [ ]:
from umbra_py import CatalogIndex, UmbraCatalog

FILTERS = dict(
    polarizations=["VV"],
    min_incidence=10,
    max_incidence=50,
    max_per_task=1,  # one pass per site keeps the sample diverse
    limit=3,
)

try:
    with CatalogIndex.from_release() as index:
        sicd_items = list(index.search(product_types=["SICD"], **FILTERS))
    print("search via CatalogIndex.from_release()")
except Exception as exc:
    # Snapshot missing or unreachable — `umbra index fetch` once, then retry.
    print(f"index snapshot unavailable ({type(exc).__name__}: {exc})")
    print("tip: run `umbra index fetch`; falling back to a tiny live crawl")
    sicd_items = list(UmbraCatalog().search(product_types=["SICD"], **FILTERS))

assert sicd_items, "no VV SICD hits in the incidence window — is the catalog reachable?"
for item in sicd_items:
    assert "SICD" in item.available_assets
    assert "VV" in item.polarizations
    assert item.incidence_angle is not None
    assert 10 <= item.incidence_angle <= 50
    print(
        item.summary(),
        "· pol:", item.polarizations,
        "· incidence (°):", round(item.incidence_angle, 1),
    )

## 2 · Size-check SICD assets (HEAD only — do not download here)

Open SICDs are often multi-GB NITFs. There is no dedicated size-confirm API:
issue an HTTP `HEAD` on each asset `href` and read `Content-Length` before you
ever call `download_item`. This notebook **does not** download the SICDs — that
would be too slow for CI / network smoke. For a real corpus, download only
after confirming size.

In [ ]:
import requests

for item in sicd_items:
    href = item.asset_href("SICD")
    resp = requests.head(href, allow_redirects=True, timeout=60)
    resp.raise_for_status()
    length = resp.headers.get("Content-Length")
    assert length is not None and int(length) > 0, f"missing Content-Length for {item.id}"
    gib = int(length) / (1024 ** 3)
    print(f"{item.id}: {gib:.2f} GiB  ({href.split('/')[-1]})")

print(
    "\nFor a real corpus: download_item(item, dest_dir=..., assets=['SICD']) "
    "after the size check above — skipped here on purpose."
)

## 3 · Fast path: chip a GEC without downloading a SICD

`GEC` is the usual chippable asset (`CHIPPABLE_ASSETS`): a geocoded COG that
`write_chips` streams window-by-window over HTTP. Same polarization / incidence
filters as above, large `chip_size` (2048, as in notebook 05) so one scene
yields a modest, quick-to-write smoke dataset. Assert the manifest and the
propagated CC-BY attribution.

In [ ]:
import json
import tempfile

from umbra_py import ATTRIBUTION, CHIPPABLE_ASSETS, write_chips

assert "GEC" in CHIPPABLE_ASSETS

gec_filters = dict(
    polarizations=["VV"],
    min_incidence=10,
    max_incidence=50,
    product_types=["GEC"],
    max_per_task=1,
    limit=1,
)

try:
    with CatalogIndex.from_release() as index:
        gec_items = list(index.search(**gec_filters))
except Exception:
    gec_items = list(UmbraCatalog().search(**gec_filters))

assert gec_items, "no VV GEC hits in the incidence window"
scene = next(it for it in gec_items if "GEC" in it.available_assets)
print(scene.summary())

out_dir = tempfile.mkdtemp(prefix="umbra-isr-chips-")
dataset = write_chips(
    [scene],
    out_dir,
    asset="GEC",
    chip_size=2048,
    min_valid=0.5,
    manifest="manifest.jsonl",
)

assert dataset.records, "the scene produced no chips"
with open(dataset.manifest_path) as fh:
    records = [json.loads(line) for line in fh]
assert len(records) == len(dataset.records)
assert all(r["attribution"] == ATTRIBUTION for r in records)
print(f"{len(dataset.records)} chips → {out_dir}")
print("attribution:", records[0]["attribution"])

## Optional · True SLC amplitude chips (SICD → COG → chip)

For training closer to ProSR's SLC-amplitude patches, convert a downloaded
SICD to a map-ready COG first (`sicd_to_geocoded_cog` / `umbra convert`), then
chip that GeoTIFF. That path needs the **`convert`** extra (`sarpy`) and is
walked end-to-end in [`07_sicd_amplitude.ipynb`](07_sicd_amplitude.ipynb).
Skip it here so this cookbook stays a fast, self-checking smoke without
multi-GB downloads.

## Where next

- **[`05_detection_chips.ipynb`](05_detection_chips.ipynb)** — chip mechanics,
  manifest fields, and opening a GeoTIFF tile as an array.
- **[`07_sicd_amplitude.ipynb`](07_sicd_amplitude.ipynb)** — SICD → geocoded COG
  when you need SLC amplitude on a map before chipping.
- **CLI** — `umbra index fetch`, then `umbra chips --local --polarizations VV
  --min-incidence 10 --max-incidence 50 --out chips/` over a whole search.
- **Docs** — [Used in research](https://umbra-py.space/guides/research/) for
  the ProSR citation and how umbra-py covers discovery → arrays (not the model).

*Contains Umbra open data, licensed under CC BY 4.0.*